有NAN  
AMT_ANNUITY: 這筆貸款的每期應繳金額，NAN就是沒有限制。  

AMT_CREDIT_MAX_OVERDUE: 這筆貸款曾經出現過的最大逾期金額，有NAN與0。  

DAYS_ENDDATE_FACT: 在申請之前，這筆帳已經結清多久，NAN表示尚未結清。  

AMT_CREDIT_SUM_LIMIT: 這筆外部貸款的信用額度上限，主要針對信用卡，但是大部分都是0。(打算忽略)  

AMT_CREDIT_SUM_DEBT: 這筆外部貸款目前尚未償還的欠款金額，有NAN但不多，0代表結清，應該會用sum來表示。  

DAYS_CREDIT_ENDDATE: 這筆貸款的「預計結束日期」，正的表示還沒結束，負的表示已經結束了(表示他欠錢逾期)。  

AMT_CREDIT_SUM: 外部金融機構貸款的核准總金額。  

----

無NAN  
CREDIT_ACTIVE: 這個債的貸款狀態，有Closed/Active/Sold/Bad debt。  

CREDIT_CURRENCY: 貨幣單位，有currency1234，幾乎都是currency1(打算忽略)。  

DAYS_CREDIT: 貸款甚麼時候發放的，-300表示300天前發放的。  

CREDIT_DAY_OVERDUE: 這筆貸款在申請日當下逾期多久，有的是好幾千的，但幾乎都是0。  

CNT_CREDIT_PROLONG: 申請延期還款的次數，大部分為0。  

AMT_CREDIT_SUM_OVERDUE: 在申請日當天的逾期金額。  

CREDIT_TYPE: 貸款類型，錢，車子，房子。  

DAYS_CREDIT_UPDATE: 資料更新的時候，-100表示100天錢更新的。  

SK_ID_BUREAU: 唯一的id。  

SK_ID_CURR: 申請者的id，可能有多個。  

----

In [1]:
import pandas as pd

In [2]:
bureau = pd.read_csv('home-credit-default-risk/bureau.csv') 
missing_rate = bureau.isnull().mean().sort_values(ascending=False) 
missing_rate_percent = (bureau.isnull().mean() * 100).sort_values(ascending=False) 
print(missing_rate_percent) 

AMT_ANNUITY               71.473490
AMT_CREDIT_MAX_OVERDUE    65.513264
DAYS_ENDDATE_FACT         36.916958
AMT_CREDIT_SUM_LIMIT      34.477415
AMT_CREDIT_SUM_DEBT       15.011932
DAYS_CREDIT_ENDDATE        6.149573
AMT_CREDIT_SUM             0.000757
CREDIT_ACTIVE              0.000000
CREDIT_CURRENCY            0.000000
DAYS_CREDIT                0.000000
CREDIT_DAY_OVERDUE         0.000000
SK_ID_BUREAU               0.000000
CNT_CREDIT_PROLONG         0.000000
AMT_CREDIT_SUM_OVERDUE     0.000000
CREDIT_TYPE                0.000000
DAYS_CREDIT_UPDATE         0.000000
SK_ID_CURR                 0.000000
dtype: float64


In [3]:
credit_type_counts = bureau["CREDIT_TYPE"].value_counts()
bureau["CREDIT_TYPE"] = bureau["CREDIT_TYPE"].apply(
    lambda x: x if credit_type_counts[x] >= 10000 else "Other"
)
print(bureau["CREDIT_TYPE"].value_counts())

CREDIT_TYPE
Consumer credit    1251615
Credit card         402195
Car loan             27690
Mortgage             18391
Microloan            12413
Other                 4124
Name: count, dtype: int64


In [4]:
nan_per_applicant = bureau.groupby("SK_ID_CURR")["DAYS_ENDDATE_FACT"].apply(lambda x: x.isna().sum())

In [5]:
import pandas as pd

# 讀取資料

# One-hot encoding 
categorical_cols = ["CREDIT_ACTIVE", "CREDIT_TYPE"]
bureau = pd.get_dummies(bureau, columns=categorical_cols, drop_first=False)

agg_dict = { 
    "AMT_CREDIT_SUM": ["mean"], 
    "AMT_CREDIT_SUM_DEBT": ["sum"], 
    "AMT_ANNUITY": ["mean"],        
    "AMT_CREDIT_MAX_OVERDUE": ["mean"], 
    "DAYS_ENDDATE_FACT": ["mean"], 
    "DAYS_CREDIT_ENDDATE": ["mean"], 
    "DAYS_CREDIT": ["mean"], 
    "CREDIT_DAY_OVERDUE": ["mean"], 
    "CNT_CREDIT_PROLONG": ["mean"], 
    "AMT_CREDIT_SUM_OVERDUE": ["sum"], 
    "DAYS_CREDIT_UPDATE": ["mean"], 
    # one-hot 後的類別型欄位，全部加總
}

# 把 one-hot 的欄位也加到 agg_dict
for col in bureau.columns:
    if col.startswith("CREDIT_ACTIVE_") or col.startswith("CREDIT_TYPE_"):
        agg_dict[col] = ["sum"]

# aggregate
bureau_agg = bureau.groupby("SK_ID_CURR").agg(agg_dict)

# # Rename columns and add 'nan_per_applicant', because NaN in this column means no repayment.
bureau_agg.columns = ["{}_{}".format(k, agg) for k, agg in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()

bureau_agg = bureau_agg.merge(
    nan_per_applicant.reset_index(name="nan_per_applicant"),
    on="SK_ID_CURR",
    how="left"
)

In [6]:
bureau_agg

,SK_ID_CURR,AMT_CREDIT_SUM_mean,AMT_CREDIT_SUM_DEBT_sum,AMT_ANNUITY_mean,AMT_CREDIT_MAX_OVERDUE_mean,DAYS_ENDDATE_FACT_mean,DAYS_CREDIT_ENDDATE_mean,DAYS_CREDIT_mean,CREDIT_DAY_OVERDUE_mean,CNT_CREDIT_PROLONG_mean,...,CREDIT_ACTIVE_Bad debt_sum,CREDIT_ACTIVE_Closed_sum,CREDIT_ACTIVE_Sold_sum,CREDIT_TYPE_Car loan_sum,CREDIT_TYPE_Consumer credit_sum,CREDIT_TYPE_Credit card_sum,CREDIT_TYPE_Microloan_sum,CREDIT_TYPE_Mortgage_sum,CREDIT_TYPE_Other_sum,nan_per_applicant
0,100001,2.076236e+05,596686.500,3545.357143,NaN,-825.500000,82.428571,-735.000000,0.0,0.000000,...,0,4,0,0,7,0,0,0,0,3
1,100002,1.081319e+05,245781.000,0.000000,1681.029,-697.500000,-349.000000,-874.000000,0.0,0.000000,...,0,6,0,0,4,4,0,0,0,2
2,100003,2.543501e+05,0.000,NaN,0.000,-1097.333333,-544.500000,-1400.750000,0.0,0.000000,...,0,3,0,0,2,2,0,0,0,1
3,100004,9.451890e+04,0.000,NaN,0.000,-532.500000,-488.500000,-867.000000,0.0,0.000000,...,0,2,0,0,2,0,0,0,0,0
4,100005,2.190420e+05,568408.500,1420.500000,0.000,-123.000000,439.333333,-190.666667,0.0,0.000000,...,0,1,0,0,2,1,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305806,456249,2.841430e+05,163071.000,NaN,6147.000,-1364.750000,-1232.333333,-1667.076923,0.0,0.000000,...,0,11,0,0,9,3,0,0,1,1
305807,456250,1.028820e+06,2232040.095,154567.965000,0.000,-760.000000,1288.333333,-862.000000,0.0,0.000000,...,0,1,0,0,2,1,0,0,0,2
305808,456253,9.900000e+05,1795833.000,58369.500000,NaN,-794.000000,280.500000,-867.500000,0.0,0.000000,...,0,2,0,0,3,1,0,0,0,2
305809,456254,4.500000e+04,0.000,0.000000,NaN,-859.000000,-859.000000,-1104.000000,0.0,0.000000,...,0,1,0,0,1,0,0,0,0,0


In [ ]:
pd.DataFrame.to_csv(bureau_agg,"transformed_data/_bureau.csv") 